In [8]:
import os
import numpy as np
import pickle
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.layers import GlobalMaxPooling2D
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input

# -----------------------
# Load model
# -----------------------
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    GlobalMaxPooling2D()
])

# -----------------------
# Config
# -----------------------
IMAGE_FOLDER = "images"
VALID_EXT = ('.jpg', '.jpeg', '.png')

# -----------------------
# Get image paths
# -----------------------
filenames = [
    os.path.join(IMAGE_FOLDER, file)
    for file in os.listdir(IMAGE_FOLDER)
    if file.lower().endswith(VALID_EXT)
]

print("Total images:", len(filenames))

# -----------------------
# Feature extraction
# -----------------------
def extract_features(img_path):
    try:
        img = image.load_img(img_path, target_size=(224,224))
        img_array = image.img_to_array(img)
        expanded_img_array = np.expand_dims(img_array, axis=0)
        preprocessed_img = preprocess_input(expanded_img_array)

        result = model.predict(preprocessed_img, verbose=0).flatten()
        normalized_result = result / np.linalg.norm(result)

        return normalized_result
    except Exception as e:
        print(f"Skipping {img_path}: {e}")
        return None

# -----------------------
# Process images
# -----------------------
feature_list = []
valid_files = []

for file in tqdm(filenames):
    features = extract_features(file)
    if features is not None:
        feature_list.append(features)
        valid_files.append(file)

feature_list = np.array(feature_list)

print("Final features shape:", feature_list.shape)

# -----------------------
# Save
# -----------------------
pickle.dump(feature_list, open('embeddings.pkl','wb'))
pickle.dump(valid_files, open('filenames.pkl','wb'))

print("Saved successfully")

Total images: 44441


100%|██████████| 44441/44441 [1:19:44<00:00,  9.29it/s]


Final features shape: (44441, 2048)
Saved successfully
